# Optimize LP01 input–output distortion by changing peak power

Vary only input peak power, starting at **100 kW**, to minimize the change
between the input pulse and nonlinear output. Keep the original **200 fs
Gaussian**, **LP01 launch and score**, **three propagated modes**, **2.86 m
step-index fiber**, and **1379.2 nm carrier**.

\[
F_{\mathrm{IO},01}=\frac{\left|\int A_{\mathrm{in},1}^{*}(0,t)
A_{\mathrm{NL},1}(L,t)\,dt\right|^2}
{\left(\int |A_{\mathrm{in},1}(0,t)|^2dt\right)
 \left(\int |A_{\mathrm{NL},1}(L,t)|^2dt\right)},
\qquad D_{\mathrm{IO},01}=1-F_{\mathrm{IO},01}.
\]

**Lower DIO is better.** Evaluate both fields on the same time samples with
**no alignment, time shift, spectral filter, or phase correction**. The complex
overlap is insensitive to a single constant complex scale factor. All three
modes propagate and may couple; this notebook retains the original LP01-only
score. Thus the modal sum in the general DIO definition is restricted to LP01.
The matched linear output appears only as a diagnostic in the final figures.

For every candidate, require **at least 50% LP01 photon retention**. Divide its unfiltered LP01 output photon count by its LP01 input photon count at the same launch power.

Run in order with **GNLSE (Julia)**. This is an independent
DIO run; its cache and saved results are separate from the source notebook.

In [ ]:
using Pkg
workspace = let dir = abspath(@__DIR__)
    while !isfile(joinpath(dir, "notebook_helpers", "run_library.py"))
        dirname(dir) == dir && error("Open this notebook inside the Nonlinear Optics workspace.")
        dir = dirname(dir)
    end
    dir
end
ROOT = get(ENV, "DIO_PROJECT_ROOT", joinpath(workspace, "FastDifferentiableMMNLO"))
Pkg.activate(ROOT)
using PulsePropagation, FFTW, LinearAlgebra, Printf, TOML, Dates, SHA, NPZ
using PyPlot

BLAS.set_num_threads(1)
FFTW.set_num_threads(1)
run_dir = haskey(ENV, "DIO_RUN_DIR") ? ENV["DIO_RUN_DIR"] :
    mktempdir(mkpath(joinpath(workspace, "output", "organized", "runs", "Total launch power", "Gradient descent")); prefix=Dates.format(now(), "yyyy-mm-dd") * "_dio_power_optimization_", cleanup=false)
mkpath(run_dir)
println("Results: ", run_dir)
plot_settings_path = get(ENV, "DIO_PLOT_SETTINGS", joinpath(workspace, "notebook_helpers", "julia", "plot_settings.toml"))

# Load the launch, objective, descent, and plotting functions used below.
include(joinpath(workspace, "notebook_helpers", "julia", "Total launch power", "shared.jl"))
include(joinpath(workspace, "notebook_helpers", "julia", "Total launch power", "dio_power_optimization.jl"))

## 1. Choose the control and compute budget

Use \(u=P/(100\,\mathrm{kW})\), **50–150 kW** bounds, and at least **50%**
of each launch’s own LP01 input photons. Search with 10 mm propagation steps; perform
one finer 1 mm propagation check for the baseline and selected result.

The finite-difference spacing remains ±200 W (clipped at the power bounds).
Each new objective evaluation runs **one nonlinear forward solve**. Keep the
original learning rate, 20-iteration limit, and 80-new-evaluation optimizer
budget. Rerun from the top after changing settings so the caches are rebuilt.

In [ ]:
reference_power_W = 100_000.0
power_bounds_W = (50_000.0, 150_000.0)
minimum_brightness_fraction = 0.50
fd_step = 0.002                  # ±200 W when differentiating with respect to u
learning_rate = 0.10
max_iterations = 20
max_optimizer_evaluations = 80
gradient_tolerance = 1e-4        # DIO per unit u
max_power_change_W = 10_000.0
search_dz = 0.01                # m
validation_dz = 0.001           # m

dio_step_tolerance = 2e-4  # absolute DIO difference, not dB
launch_coefficients = [1.0, 0.0, 0.0]

settings = Dict(
    "control" => "input Gaussian peak power (W)",
    "objective" => "minimize LP01 DIO = 1 - input/nonlinear-output complex field fidelity",
    "reference_power_W" => reference_power_W,
    "power_bounds_W" => collect(power_bounds_W),
    "minimum_brightness_fraction" => minimum_brightness_fraction,
    "finite_difference_step_u" => fd_step,
    "learning_rate" => learning_rate,
    "max_iterations" => max_iterations,
    "max_optimizer_evaluations" => max_optimizer_evaluations,
    "gradient_tolerance" => gradient_tolerance,
    "max_power_change_W" => max_power_change_W,
    "search_dz_m" => search_dz,
    "validation_dz_m" => validation_dz,
    "Nt" => 2048, "time_window_ps" => 10.0, "pulse_fwhm_ps" => 0.2,
    "fiber_length_m" => 2.86, "carrier_m" => 1379.2e-9,
    "core_radius_m" => 52.5e-6, "NA" => 0.1, "num_modes" => 3,
    "objective_mode" => 1, "objective_filter" => "none",
    "objective_reference" => "original candidate input on unchanged time samples; no alignment or shift",
    "linear_diagnostic" => "same candidate input and fiber; n2 = 0; final plots only",
    "brightness_definition" => "unfiltered LP01 output photons divided by LP01 input photons at the same power",
    "dio_step_tolerance" => dio_step_tolerance,
    "launch_coefficients" => launch_coefficients,
    "cp_target_error" => 1e-3, "cp_seed" => Int(0x4d4d474e),
    "julia_version" => string(VERSION),
    "manifest_sha256" => bytes2hex(sha256(read(joinpath(ROOT, "Manifest.toml")))),
)
open(joinpath(run_dir, "settings.toml"), "w") do io
    TOML.print(io, settings)
end

## 2. Build the fixed fiber

Use the original three-mode fiber and CP compression. The nonlinear parameter
set supplies the objective; the n2 = 0 parameter set is used only for final
baseline/selected diagnostic plots. No dense-versus-CP or extra time-grid sweep
is run here.

Photon retention counts LP01 photons over the full spectrum at the input and output. All-mode photon count
and modal energy fractions remain physical diagnostics.

In [ ]:
time_grid = TimeGrid(settings["Nt"], settings["time_window_ps"])
domain = MMGNLSEDomain(degrees_of_freedom(:time, :space), time_grid)
properties = compute_fiber_properties(
    StepIndex(settings["carrier_m"], settings["core_radius_m"], settings["NA"]),
    Silica(), settings["num_modes"];
    polarization=:scalar, mode_basis=:LP, beta_order=5)
S_cp = cp_compress(properties.S; error=settings["cp_target_error"], seed=settings["cp_seed"])
parameters_cp = MMGNLSEParameters(domain;
    length=settings["fiber_length_m"], beta=properties.beta, S=S_cp,
    n2=properties.n2, omega0=properties.omega0, raman=properties.raman)


parameters_linear = MMGNLSEParameters(domain;
    length=settings["fiber_length_m"], beta=properties.beta, S=S_cp,
    n2=0.0, omega0=properties.omega0, raman=properties.raman)
lp01_photons = SpectralPhotonNumber(filter=ones(settings["Nt"]), modes=[1])
all_photons = SpectralPhotonNumber(filter=ones(settings["Nt"]), modes=:all)
@printf("Three propagated modes; LP01 launch and score. CP error: %.3g\n", S_cp.relative_error)
println("Mode labels: ", properties.modes.labels)

## 3. Turn the DIO calculation into a function

The common time-spacing factors cancel in the normalized overlap. Compare the
**original input and nonlinear output**, including their complex phase, at
identical time indices. Keep the original LP01-only score and reject zero or
nonfinite field norms. Algebra checks establish self-overlap, invariance to a
constant complex factor, and zero overlap for orthogonal samples.

Cache each nonlinear evaluation and its endpoint fields in this independent
run directory. Linear propagation is deferred until the final plots.

In [ ]:
evaluation_cache = Dict{Tuple{Float64,Float64},NamedTuple}()
field_cache = Dict{Tuple{Float64,Float64},NamedTuple}()
evaluations = NamedTuple[]

let a = zeros(ComplexF64, 4, 3, 1)
    a[1, 1, 1] = 1
    @assert isapprox(lp01_fidelity(a, a), 1.0; atol=1e-12)
    @assert isapprox(lp01_fidelity(a, 2exp(0.7im).*a), 1.0; atol=1e-12)
    b = copy(a)
    b[:, 2, 1] .= 3  # Changes in other modes do not enter this objective.
    @assert isapprox(lp01_fidelity(a, b), 1.0; atol=1e-12)
    b = zeros(ComplexF64, size(a))
    b[2, 1, 1] = 1
    @assert lp01_fidelity(a, b) == 0.0
    rejected_zero = try
        lp01_fidelity(a, zero(a))
        false
    catch err
        err isa ArgumentError
    end
    @assert rejected_zero
end
println("LP01 overlap definition checks passed.")

## 4. Check the starting pulse

Evaluate the 100 kW starting pulse with 10 mm and 1 mm propagation steps. Each candidate uses its own input photon count as the denominator in the retention test.

In [ ]:
baseline = dio_at(reference_power_W)
baseline_fine = dio_at(reference_power_W; dz=validation_dz)
baseline_step_error = abs(baseline.D_IO-baseline_fine.D_IO)
@assert baseline_step_error < dio_step_tolerance "Reduce search_dz before optimization."
@assert baseline_fine.photon_retention >= minimum_brightness_fraction
@printf("10 mm versus 1 mm baseline DIO difference: %.8f\n", baseline_step_error)
@printf("Minimum LP01 output/input photon retention: %.1f%%\n", 100minimum_brightness_fraction)

## 5. Projected gradient descent with backtracking

Keep the original algorithm: propose \(u_{\rm new}=\operatorname{clip}(u-\eta g)\),
cap the power change, and halve it if the trial loses too many LP01 photons or
does not decrease DIO sufficiently. Save accepted steps only.

Stop on a small projected gradient, failed line search, or the iteration/evaluation
budget. A budget stop does not establish a minimum.

In [ ]:
optimization = optimize_power()
println("Stop: ", optimization.status)
@printf("Best accepted power: %.4f kW; objective: DIO=%.8f\n",
    optimization.best.power_W/1000, optimization.best.D_IO)

## 6. Validate the selected result once

Compare baseline and selected DIO at the same finer **1 mm** propagation step.
Check power bounds, finite physical observables, LP01 brightness, and agreement
with the search step. Report the measured change even if the starting point
remains best. No extra neighbor scan, intermediate propagation step, dense-CP
comparison, or doubled time-grid test is added.

The reported stop status describes this bounded search. An iteration/evaluation
budget stop does not establish a converged or global minimum.

In [ ]:
best_power_W = optimization.best.power_W
best_fine = dio_at(best_power_W; dz=validation_dz)
improvement_DIO = baseline_fine.D_IO-best_fine.D_IO
best_step_error = abs(optimization.best.D_IO-best_fine.D_IO)
@assert power_bounds_W[1] <= best_power_W <= power_bounds_W[2]
@assert best_fine.photon_retention >= minimum_brightness_fraction "Photon retention constraint fails on the finer propagation step."
@assert best_step_error < dio_step_tolerance "Refine search_dz before interpreting this result."
for row in (baseline_fine, best_fine)
    @assert isfinite(row.D_IO) && 0 <= row.D_IO <= 1
    @assert isfinite(row.photons) && row.photons > 0
    @assert isfinite(row.total_photons) && row.total_photons > 0
    @assert isfinite(row.output_input_energy_ratio) && row.output_input_energy_ratio > 0
    @assert 0 <= row.lp01_energy_fraction <= 1
end

summary = Dict(
    "status"=>optimization.status,
    "baseline_power_W"=>reference_power_W, "best_power_W"=>best_power_W,
    "baseline_D_IO"=>baseline_fine.D_IO, "best_D_IO"=>best_fine.D_IO,
    "baseline_F_IO"=>baseline_fine.F_IO, "best_F_IO"=>best_fine.F_IO,
    "improvement_DIO"=>improvement_DIO,
    "relative_DIO_reduction"=>1-best_fine.D_IO/baseline_fine.D_IO,
    "photon_reference"=>settings["brightness_definition"],
    "baseline_input_photons"=>baseline_fine.input_photons, "best_input_photons"=>best_fine.input_photons,
    "baseline_photon_retention"=>baseline_fine.photon_retention,
    "baseline_photons"=>baseline_fine.photons, "best_photons"=>best_fine.photons,
    "brightness_ratio"=>best_fine.photon_retention,
    "best_lp01_energy_fraction"=>best_fine.lp01_energy_fraction,
    "best_mode2_energy_fraction"=>best_fine.mode2_energy_fraction,
    "best_mode3_energy_fraction"=>best_fine.mode3_energy_fraction,
    "baseline_output_input_energy_ratio"=>baseline_fine.output_input_energy_ratio,
    "best_output_input_energy_ratio"=>best_fine.output_input_energy_ratio,
    "baseline_step_error"=>baseline_step_error, "best_step_error"=>best_step_error,
    "accepted_steps"=>length(optimization.history)-1,
    "optimization_seconds"=>optimization.seconds,
    "new_optimizer_objective_evaluations"=>optimization.new_objective_evaluations,
    "cp_relative_error"=>S_cp.relative_error,
    "validation_passed"=>true,
    "validation_scope"=>"LP01 objective algebra, physical feasibility/brightness, one finer-propagation baseline/selected check",
    "objective_alignment"=>"same time samples; no shifting or alignment",
)
open(joinpath(run_dir, "summary.toml"), "w") do io
    TOML.print(io, summary)
end
@printf("Selected: %.3f → %.3f kW; finer DIO %.8f → %.8f\n",
    reference_power_W/1000, best_power_W/1000, baseline_fine.D_IO, best_fine.D_IO)
@printf("DIO reduction: %.2f%%; LP01 input photons retained: %.2f%%\n",
    100summary["relative_DIO_reduction"], 100summary["brightness_ratio"])
@printf("Optimization: %.1f s, %d accepted steps, %d new objective evaluations\n",
    optimization.seconds, summary["accepted_steps"], optimization.new_objective_evaluations)
println("Stop: ", optimization.status)

## 7. Look at the power dependence and optimization path

Keep the original diagnostic sweep over 51 equally spaced powers from
50–150 kW (2 kW spacing), overlaid with the accepted DIO descent. The sweep is
evaluated after optimization and does not select the result. Each new sweep
point needs one nonlinear propagation. Crosses identify the 1 mm baseline
and selected validation values; sweep/search values use 10 mm steps.

In [ ]:
scan_lower, scan_upper = power_bounds_W
scan = [dio_at(P) for P in range(scan_lower, scan_upper; length=51)]
save_rows("power_sweep.csv", scan)

fig, axes = subplots(1, 2; figsize=(9, 3.5), constrained_layout=true)
axes[1].plot([r.power_W/1000 for r in scan], [r.D_IO for r in scan],
    "o-"; color="#366b9e", label="Full power sweep")
axes[1].plot([r.power_W/1000 for r in optimization.history],
    [r.D_IO for r in optimization.history], "s--";
    color="#cf7c32", label="Accepted gradient steps", markersize=4)
axes[1].scatter([reference_power_W/1000, best_power_W/1000],
    [baseline_fine.D_IO, best_fine.D_IO];
    color="black", marker="x", s=60, label="1 mm validation", zorder=5)
axes[1].set_xlabel("Input peak power (kW)")
axes[1].set_ylabel("LP01 input–output distortion DIO")
axes[1].set_title("One input: peak power")
axes[1].legend(; fontsize=8)
axes[2].plot([r.iteration for r in optimization.history],
    [r.D_IO for r in optimization.history], "o-"; color="#366b9e")
axes[2].set_xlabel("Accepted iteration")
axes[2].set_ylabel("LP01 input–output distortion DIO")
axes[2].set_title("Gradient descent: lower is better")
for ax in axes
    ax.grid(; alpha=0.2)
    ax.spines["top"].set_visible(false)
    ax.spines["right"].set_visible(false)
end
savefig(joinpath(run_dir, "dio_power_optimization.png"); dpi=180)
savefig(joinpath(run_dir, "dio_power_optimization.pdf"))
display(fig)
close(fig)
@printf("Total objective evaluations (setup, search, validation, plot): %d\n", length(evaluations))
@printf("Total measured objective time: %.1f s\n", sum(r.seconds for r in evaluations))

## 8. Inspect the baseline and selected LP01 output

For each input power, show the time-domain pulse and the wavelength spectrum:

- Time (ps) versus LP01 power (kW).
- Wavelength (nm) versus power per frequency bin (W), matching
  step_index_squeezing.ipynb.

The spectrum plots frequency-bin powers at their associated wavelengths.
Its vertical axis is **not energy per nanometre**. Input, matched linear
output, and nonlinear output share the same time samples; output propagation
uses 1 mm steps. The linear branch is computed only for these diagnostics. The curves use
absolute units. Display limits zoom to occupied support at a relative threshold
of 10⁻⁵; full fields remain in the NPZ files.

In [ ]:
times_ps = collect(time_axis(time_grid))
detuning_THz = collect(frequency_axis(time_grid)) ./ (2pi)
wavelengths_nm = collect(wavelength_axis(time_grid, properties.omega0)) .* 1e9
wavelength_indices = findall(isfinite.(wavelengths_nm) .& (wavelengths_nm .> 0))
wavelength_indices = wavelength_indices[sortperm(wavelengths_nm[wavelength_indices])]

plot_lp01_output("baseline", baseline_fine)
plot_lp01_output("selected", best_fine)
println("Saved time plots, wavelength spectra, and endpoint fields to ", run_dir)

## 9. Take a peek at the spatial modes, input and output

First, display the three fixed mode shapes in the same style as the original
notebook. Red and blue indicate opposite signs of the mode's field amplitude.
These shapes form the spatial basis used to propagate the pulse.
The dashed circle is the fiber core; both axes are positions in micrometres.

In [ ]:
x = properties.modes.x
y = properties.modes.y
modes = properties.modes.fields  # axes: y, x, mode
mode_names = ("LP01", "LP11 — cos orientation", "LP11 — sin orientation")

let fig = figure(; figsize=(18, 6), constrained_layout=true)
    for ii = 1:3
        ax = subplot(1, 3, ii)
        # Normalize amplitude only for displaying the fixed mode shapes.
        profile = real.(modes[:, :, ii])
        profile = profile ./ maximum(abs, profile)
        mesh = ax.pcolormesh(x.*1e6, y.*1e6, profile;
            cmap="seismic", vmin=-1, vmax=1, shading="auto", rasterized=true)
        ax.set_title("Mode $ii: "*mode_names[ii])
        ax.set_xlabel("x (μm)")
        ax.set_ylabel("y (μm)")
        ax.set_aspect("equal")
        theta = range(0, 2pi; length=181)
        ax.plot(settings["core_radius_m"].*1e6.*cos.(theta),
            settings["core_radius_m"].*1e6.*sin.(theta);
            color="#555555", linestyle="--", linewidth=0.8)
        fig.colorbar(mesh; ax, shrink=0.75, label="Relative field amplitude")
    end
    fig.suptitle("The three spatial modes — dashed line marks the fiber core")
    savefig(joinpath(run_dir, "spatial_modes.png"); dpi=160)
    savefig(joinpath(run_dir, "spatial_modes.pdf"))
    display(fig)
    close(fig)
end
npzwrite(joinpath(run_dir, "spatial_mode_fields.npz"),
    Dict("x_m"=>x, "y_m"=>y, "modes_y_x_m"=>modes))

Combine the saved complex mode amplitudes with the spatial modes to display
**input, linear-output, and nonlinear-output intensity**. All three propagated
modes contribute, although the DIO objective scores only LP01.

Each panel shows its pulse at **its own peak total-power time**, printed in
the title. Every physical beam or mode intensity panel reads the same absolute
color limits and colormap from **plot_settings.toml**, shared across the DIO
notebooks. Complex modal amplitudes are summed before squaring, preserving
interference. The integrated beam map is checked against the pulse power.

These maps use cached endpoint fields. The maximum displayed intensity is
recorded so a single common color limit can cover every notebook without
clipping. Signed basis-amplitude maps above retain the separate −1…1 scale.

In [ ]:
# Read the common absolute intensity scale for all DIO notebooks.
plot_settings = TOML.parsefile(plot_settings_path)
beam_intensity_limits_W_per_um2 = (Float64(plot_settings["beam_intensity_min_W_per_um2"]),
    Float64(plot_settings["beam_intensity_max_W_per_um2"]))
beam_colormap = plot_settings["colormap"]
displayed_intensity_maxima = Dict{String,Float64}()
beam_color_ticks = collect(range(beam_intensity_limits_W_per_um2...; length=6))

plot_beam_preview("baseline", baseline_fine)
plot_beam_preview("selected", best_fine)
plot_individual_mode_intensity("baseline", baseline_fine)
plot_individual_mode_intensity("selected", best_fine)
beam_report = Dict("max_displayed_intensity_W_per_um2"=>maximum(values(displayed_intensity_maxima)),
    "maxima"=>displayed_intensity_maxima, "plot_settings_path"=>plot_settings_path,
    "shared_color_max_W_per_um2"=>beam_intensity_limits_W_per_um2[2])
open(joinpath(run_dir, "beam_intensity_report.toml"), "w") do io
    TOML.print(io, beam_report)
end
@printf("Highest displayed beam/mode intensity: %.8f W/μm²\n", beam_report["max_displayed_intensity_W_per_um2"])